# 09: GTFS/OSM/Census Pull for the New NWSL Venues

Data pull for the 9 current NWSL venues that have no isochrone coverage yet (everything
outside the 5 metros 01/02/06 already built: Gotham, KC Current, San Diego Wave, Seattle
Reign, Washington Spirit). Same method and inputs as `01_gtfs_pull_comparison_cities.ipynb`,
just a later pass because didn't initially plan for this scope.

Denver Summit FC and Boston Legacy FC (both 2026 expansion) are excluded because no solid data.

Pulls, per metro:
1. GTFS feed(s) for the relevant transit agencies.
2. Geofabrik state .osm.pbf extract(s) (California already downloaded for San Diego).
3. Census TIGER tract shapefile(s) for each state touched.
4. osmium clip of each state extract down to a metro bbox (keeps r5py network builds fast).

Run once; the r5py network build happens per-metro in 08_isochrones_new_venues.py, launched in parallel.

In [1]:
import os
import shutil
import subprocess

import requests

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
GTFS_DIR = os.path.join("..", "data", "raw", "gtfs")
OSM_DIR = os.path.join("..", "data", "raw", "osm")
CENSUS_DIR = os.path.join("..", "data", "raw", "census")
for d in (GTFS_DIR, OSM_DIR, CENSUS_DIR):
    os.makedirs(d, exist_ok=True)


def download_file(url, dest_path, timeout=900, retries=3):
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        print(f"  [skip, exists] {dest_path}")
        return
    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"}) as r:
                r.raise_for_status()
                tmp = dest_path + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                os.rename(tmp, dest_path)
            print(f"  [ok] {dest_path} ({os.path.getsize(dest_path) / 1e6:.1f} MB)")
            return
        except Exception as e:
            print(f"  [retry {attempt}/{retries}] {url} -> {e}")
    raise RuntimeError(f"Failed to download {url} after {retries} attempts")


# GTFS feeds
GTFS_FEEDS = {
    # San Jose (Bay FC)
    "vta.zip": "https://gtfs.vta.org/gtfs_vta.zip",
    "caltrain.zip": "https://data.trilliumtransit.com/gtfs/caltrain-ca-us/caltrain-ca-us.zip",
    # Los Angeles (Angel City FC)
    "lametro_bus.zip": "https://gitlab.com/LACMTA/gtfs_bus/-/raw/master/gtfs_bus.zip",
    # Houston (Houston Dash)
    "houston_metro.zip": "https://files.mobilitydatabase.org/mdb-2060/mdb-2060-202607120126/mdb-2060-202607120126.zip",
    # Chicago / Evanston (Chicago Stars FC, Northwestern Stadium)
    "cta.zip": "https://www.transitchicago.com/downloads/sch_data/google_transit.zip",
    "metra.zip": "https://schedules.metrarail.com/gtfs/schedule.zip",
    # Portland (Portland Thorns)
    "trimet.zip": "https://developer.trimet.org/schedule/gtfs.zip",
    # Orlando (Orlando Pride)
    "lynx.zip": "https://files.mobilitydatabase.org/mdb-347/mdb-347-202607300104/mdb-347-202607300104.zip",
    # Salt Lake City / Sandy (Utah Royals FC)
    "uta.zip": "https://gtfsfeed.rideuta.com/gtfs.zip",
    # Louisville (Racing Louisville FC)
    "tarc.zip": "https://tarc.rideralerts.com/InfoPoint/gtfs-zip.ashx",
    # Raleigh/Cary (North Carolina Courage)
    "gotriangle.zip": "http://data.trilliumtransit.com/gtfs/tta-regionalbus-nc-us/tta-regionalbus-nc-us.zip",
}

In [2]:
print("=== GTFS feeds ===")
for name, url in GTFS_FEEDS.items():
    print(f"{name}: {url}")
    download_file(url, os.path.join(GTFS_DIR, name))

# Geofabrik state OSM extracts. california-latest.osm.pbf already exists 
OSM_EXTRACTS = {
    "texas-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/texas-latest.osm.pbf",
    "illinois-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/illinois-latest.osm.pbf",
    "oregon-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/oregon-latest.osm.pbf",
    "florida-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/florida-latest.osm.pbf",
    "utah-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/utah-latest.osm.pbf",
    "kentucky-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/kentucky-latest.osm.pbf",
    "north-carolina-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/north-carolina-latest.osm.pbf",
}

=== GTFS feeds ===
vta.zip: https://gtfs.vta.org/gtfs_vta.zip
  [skip, exists] ../data/raw/gtfs/vta.zip
caltrain.zip: https://data.trilliumtransit.com/gtfs/caltrain-ca-us/caltrain-ca-us.zip
  [skip, exists] ../data/raw/gtfs/caltrain.zip
lametro_bus.zip: https://gitlab.com/LACMTA/gtfs_bus/-/raw/master/gtfs_bus.zip
  [skip, exists] ../data/raw/gtfs/lametro_bus.zip
houston_metro.zip: https://files.mobilitydatabase.org/mdb-2060/mdb-2060-202607120126/mdb-2060-202607120126.zip
  [skip, exists] ../data/raw/gtfs/houston_metro.zip
cta.zip: https://www.transitchicago.com/downloads/sch_data/google_transit.zip
  [skip, exists] ../data/raw/gtfs/cta.zip
metra.zip: https://schedules.metrarail.com/gtfs/schedule.zip
  [skip, exists] ../data/raw/gtfs/metra.zip
trimet.zip: https://developer.trimet.org/schedule/gtfs.zip
  [skip, exists] ../data/raw/gtfs/trimet.zip
lynx.zip: https://files.mobilitydatabase.org/mdb-347/mdb-347-202607300104/mdb-347-202607300104.zip
  [skip, exists] ../data/raw/gtfs/lynx.zip
u

In [3]:
print("=== OSM state extracts ===")
for name, url in OSM_EXTRACTS.items():
    print(f"{name}: {url}")
    download_file(url, os.path.join(OSM_DIR, name), timeout=1800)

# Census TIGER tract shapefiles
TIGER_TRACTS = {
    "tl_2023_48_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_48_tract.zip",  # TX
    "tl_2023_17_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_17_tract.zip",  # IL
    "tl_2023_41_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_41_tract.zip",  # OR
    "tl_2023_12_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_12_tract.zip",  # FL
    "tl_2023_49_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_49_tract.zip",  # UT
    "tl_2023_21_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_21_tract.zip",  # KY
    "tl_2023_37_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_37_tract.zip",  # NC
}

=== OSM state extracts ===
texas-latest.osm.pbf: https://download.geofabrik.de/north-america/us/texas-latest.osm.pbf
  [skip, exists] ../data/raw/osm/texas-latest.osm.pbf
illinois-latest.osm.pbf: https://download.geofabrik.de/north-america/us/illinois-latest.osm.pbf
  [skip, exists] ../data/raw/osm/illinois-latest.osm.pbf
oregon-latest.osm.pbf: https://download.geofabrik.de/north-america/us/oregon-latest.osm.pbf
  [skip, exists] ../data/raw/osm/oregon-latest.osm.pbf
florida-latest.osm.pbf: https://download.geofabrik.de/north-america/us/florida-latest.osm.pbf
  [skip, exists] ../data/raw/osm/florida-latest.osm.pbf
utah-latest.osm.pbf: https://download.geofabrik.de/north-america/us/utah-latest.osm.pbf
  [skip, exists] ../data/raw/osm/utah-latest.osm.pbf
kentucky-latest.osm.pbf: https://download.geofabrik.de/north-america/us/kentucky-latest.osm.pbf
  [skip, exists] ../data/raw/osm/kentucky-latest.osm.pbf
north-carolina-latest.osm.pbf: https://download.geofabrik.de/north-america/us/north-c

In [4]:
print("=== Census TIGER tracts ===")
for name, url in TIGER_TRACTS.items():
    print(f"{name}: {url}")
    download_file(url, os.path.join(CENSUS_DIR, name))

# osmium clip each state extract to its metro bbox 
# crop to be able to fit around the 60 minute isochromes
METRO_BBOXES = {
    "san-jose-metro.osm.pbf": ("california-latest.osm.pbf", (-122.15, 37.10, -121.70, 37.55)),
    "los-angeles-metro.osm.pbf": ("california-latest.osm.pbf", (-118.55, 33.85, -118.05, 34.25)),
    "houston-metro.osm.pbf": ("texas-latest.osm.pbf", (-95.65, 29.55, -95.05, 29.95)),
    "chicago-metro.osm.pbf": ("illinois-latest.osm.pbf", (-87.95, 41.85, -87.45, 42.25)),
    "portland-metro.osm.pbf": ("oregon-latest.osm.pbf", (-122.95, 45.30, -122.40, 45.70)),
    "orlando-metro.osm.pbf": ("florida-latest.osm.pbf", (-81.65, 28.30, -81.10, 28.70)),
    "salt-lake-metro.osm.pbf": ("utah-latest.osm.pbf", (-112.10, 40.40, -111.70, 40.80)),
    "louisville-metro.osm.pbf": ("kentucky-latest.osm.pbf", (-85.95, 38.05, -85.45, 38.45)),
    "raleigh-cary-metro.osm.pbf": ("north-carolina-latest.osm.pbf", (-78.95, 35.65, -78.55, 36.05)),
}

=== Census TIGER tracts ===
tl_2023_48_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_48_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_48_tract.zip
tl_2023_17_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_17_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_17_tract.zip
tl_2023_41_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_41_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_41_tract.zip
tl_2023_12_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_12_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_12_tract.zip
tl_2023_49_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_49_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_49_tract.zip
tl_2023_21_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_21_tract.zip
  [skip, exists] ../data/raw/census/tl_2023_21_tract.zip
tl_2023_37_tract.zip: https://www2.census.gov/geo/tiger/TIGER2023/TRACT/

In [5]:
print("=== osmium clip to metro bboxes ===")
if shutil.which("osmium") is None:
    print("osmium CLI not found -> install with `brew install osmium-tool`, then re-run this script.")
else:
    for clipped_name, (source_name, bbox) in METRO_BBOXES.items():
        clipped_path = os.path.join(OSM_DIR, clipped_name)
        source_path = os.path.join(OSM_DIR, source_name)
        if os.path.exists(clipped_path):
            print(f"  [skip, exists] {clipped_path}")
            continue
        bbox_str = ",".join(str(v) for v in bbox)
        print(f"  clipping {source_name} -> {clipped_name} (bbox {bbox_str})")
        subprocess.run(["osmium", "extract", "--bbox", bbox_str, "-o", clipped_path, source_path, "-O"], check=True)

print("\nDONE -- all downloads + clips complete.")

=== osmium clip to metro bboxes ===
  [skip, exists] ../data/raw/osm/san-jose-metro.osm.pbf
  [skip, exists] ../data/raw/osm/los-angeles-metro.osm.pbf
  [skip, exists] ../data/raw/osm/houston-metro.osm.pbf
  [skip, exists] ../data/raw/osm/chicago-metro.osm.pbf
  [skip, exists] ../data/raw/osm/portland-metro.osm.pbf
  [skip, exists] ../data/raw/osm/orlando-metro.osm.pbf
  [skip, exists] ../data/raw/osm/salt-lake-metro.osm.pbf
  [skip, exists] ../data/raw/osm/louisville-metro.osm.pbf
  [skip, exists] ../data/raw/osm/raleigh-cary-metro.osm.pbf

DONE -- all downloads + clips complete.
